# Spaceship Titanic XGBoost Demo: Clean Reproduction

This notebook is a compact classroom-demo version of `0-814-optuna-xgb-space-titanic.ipynb`.

Goal: remove EDA/noisy display cells and keep only the reproducible training path:

1. Load official `train.csv`, `test.csv`, and `sample_submission.csv`.
2. Rebuild the original compact feature engineering branch.
3. Use the fixed XGBoost parameters found by the original Optuna-style search.
4. Run a short validation / parameter sanity check for demo evidence.
5. Train the final XGBoost model with KMeansSMOTE and write `Submission_XGB_demo.csv`.

Note: the original notebook computes an IsolationForest outlier diagnostic. Its final prediction path continues from the full pruned feature matrix, so this demo keeps that behavior by default for faithful reproduction.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import KMeansSMOTE
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils import shuffle
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


## 1. Configuration

The paths below work in this local project. If this notebook is moved, update `DATA_DIR` only.


In [ ]:
def find_project_paths() -> tuple[Path, Path]:
    """Find data whether CSVs are in ./data or directly beside this notebook."""
    for root in [Path.cwd(), *Path.cwd().parents]:
        for data_dir in [root / "data", root]:
            if (data_dir / "train.csv").exists() and (data_dir / "test.csv").exists():
                return root, data_dir
    raise FileNotFoundError("Could not find train.csv/test.csv in the current directory or a data/ folder")


ROOT, DATA_DIR = find_project_paths()

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
OUTPUT_PATH = ROOT / "submissions" / "Submission_XGB_demo.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

XGB_BEST_PARAMS = {
    "reg_lambda": 3.06,
    "reg_alpha": 4.582,
    "colsample_bytree": 0.93,
    "subsample": 0.96,
    "n_estimators": 725,
    "max_depth": 5,
    "learning_rate": 0.05,
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
    "eval_metric": "logloss",
    "verbosity": 0,
}

DROP_LIST = [
    "ShoppingMall",
    "Age",
    "CryoSleep_True",
    "HomePlanet_Earth",
    "HomePlanet_Europa",
    "VIP_True",
    "HomePlanet_Mars",
    "Destination_PSO J318.5-22",
    "VIP_False",
    "Destination_55 Cancri e",
    "FoodCourt",
    "Destination_TRAPPIST-1e",
]

print(f"Project root: {ROOT.name}")
print(f"Data location: {DATA_DIR.name if DATA_DIR != ROOT else 'project root'}")


## 2. Load Data


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"train: {train_raw.shape}")
print(f"test:  {test_raw.shape}")
print(f"sample_submission: {sample_submission.shape}")
display(train_raw.head(3))


## 3. Compact Feature Engineering

This function keeps the transformations that matter for the high-scoring XGB branch:

- set spending columns to zero for passengers in CryoSleep;
- infer missing CryoSleep when total expenses are zero;
- fill group-level missing values using the PassengerId room/group prefix;
- split Cabin into deck / number / side;
- mean-impute numeric columns and mode-impute categorical columns;
- one-hot encode the compact categorical set.


In [ ]:
def make_one_hot_encoder() -> OneHotEncoder:
    """Support both new and old scikit-learn keyword names."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_compact_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    all_df = pd.concat([train_df, test_df], ignore_index=True)

    all_df.loc[all_df["CryoSleep"].eq(True), SPEND_COLS] = 0
    all_df["Expenses"] = all_df[SPEND_COLS].sum(axis=1)
    all_df.loc[(all_df["Expenses"].eq(0)) & all_df["CryoSleep"].isna(), "CryoSleep"] = True

    all_df["Name"] = all_df["Name"].fillna("Unknown Unknown")
    all_df["Room"] = all_df["PassengerId"].astype(str).str.slice(0, 4)

    for col in ["VIP", "Cabin", "HomePlanet", "Destination"]:
        guide = all_df[["Room", col]].dropna().drop_duplicates("Room").set_index("Room")[col]
        all_df[col] = all_df[col].fillna(all_df["Room"].map(guide))

    cabin_split = all_df["Cabin"].str.split("/", expand=True)
    all_df["Cabin_1"] = cabin_split[0]
    all_df["Cabin_2"] = cabin_split[1]
    all_df["Cabin_3"] = cabin_split[2]

    name_split = all_df["Name"].str.split(" ", expand=True)
    all_df["FirstName"] = name_split[0]
    all_df["SecondName"] = name_split[1]
    all_df["Name_key"] = all_df["SecondName"] + all_df["Room"]

    num_cols = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Expenses", "Age"]
    cat_cols = ["CryoSleep", "Cabin_1", "Cabin_3", "VIP", "HomePlanet", "Destination"]
    compact = all_df[num_cols + cat_cols + ["Transported"]].copy()

    compact[num_cols] = pd.DataFrame(
        SimpleImputer(strategy="mean").fit_transform(compact[num_cols]),
        columns=num_cols,
    )
    compact[cat_cols] = pd.DataFrame(
        SimpleImputer(strategy="most_frequent").fit_transform(compact[cat_cols]),
        columns=cat_cols,
    )

    ohe = make_one_hot_encoder()
    encoded = pd.DataFrame(ohe.fit_transform(compact[cat_cols]), columns=ohe.get_feature_names_out())
    compact = pd.concat(
        [compact.drop(columns=cat_cols).reset_index(drop=True), encoded.reset_index(drop=True)],
        axis=1,
    )

    train_proc = compact[compact["Transported"].notna()].copy()
    train_proc["Transported"] = train_proc["Transported"].astype(int)
    test_proc = compact[compact["Transported"].isna()].drop(columns="Transported").copy()

    X = train_proc.drop(columns="Transported")
    y = train_proc["Transported"]
    groups = train_df["PassengerId"].astype(str).str.split("_").str[0].reset_index(drop=True)
    return X, y, test_proc, groups


X_full, y, test_full, groups = build_compact_features(train_raw, test_raw)
print(f"Feature matrix before pruning: X={X_full.shape}, test={test_full.shape}")
display(X_full.head(3))


## 4. Validation Snapshot

For a live demo, the first score uses 3-fold CV so it finishes quickly. Increase `n_splits` to 10 if you want to mirror the long original notebook audit.


In [ ]:
def cv_accuracy(model_params: dict, X: pd.DataFrame, y: pd.Series, n_splits: int = 3) -> tuple[float, float]:
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(
        XGBClassifier(**model_params),
        X,
        y,
        scoring="accuracy",
        cv=splitter,
    )
    return float(scores.mean()), float(scores.std())


base_mean, base_std = cv_accuracy(XGB_BEST_PARAMS, X_full, y, n_splits=3)
print(f"Before feature pruning, 3-fold CV accuracy = {base_mean:.5f} +/- {base_std:.5f}")


## 5. IsolationForest Diagnostic and Feature Pruning

The original notebook diagnosed outliers with IsolationForest, then used the pruned feature set for the final model. We report the retained row count here so the demo stays auditable.


In [ ]:
features_isolation = ["ShoppingMall", "FoodCourt", "RoomService", "Spa", "VRDeck", "Age"]
isolation = IsolationForest(
    n_jobs=-1,
    random_state=1,
    n_estimators=100,
    contamination=0.003,
)
keep_mask = isolation.fit_predict(X_full[features_isolation]) == 1
print(f"IsolationForest retained rows: {int(keep_mask.sum())} / {len(keep_mask)}")

X_model = X_full.drop(columns=DROP_LIST)
test_model = test_full.drop(columns=DROP_LIST)
print(f"Feature matrix after pruning: X={X_model.shape}, test={test_model.shape}")
print("Remaining features:")
print(list(X_model.columns))


## 6. Lightweight Parameter Check

This is the small extra experiment added for the demo. It is not meant to replace the original Optuna search; it just shows that nearby XGB choices were compared under the same compact feature branch.


In [ ]:
RUN_EXTRA_PARAM_CHECK = True

if RUN_EXTRA_PARAM_CHECK:
    candidates = {
        "final_params": {},
        "500_trees": {"n_estimators": 500},
        "900_trees": {"n_estimators": 900},
        "depth_4": {"max_depth": 4},
        "depth_6": {"max_depth": 6},
        "lr_0.06": {"learning_rate": 0.06, "n_estimators": 600},
    }

    rows = []
    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    for name, override in candidates.items():
        params = {**XGB_BEST_PARAMS, **override}
        fold_scores = []
        for train_idx, valid_idx in splitter.split(X_model, y, groups=groups):
            model = XGBClassifier(**params)
            model.fit(X_model.iloc[train_idx], y.iloc[train_idx])
            pred = model.predict(X_model.iloc[valid_idx])
            fold_scores.append(accuracy_score(y.iloc[valid_idx], pred))
        rows.append(
            {
                "candidate": name,
                "mean_accuracy": np.mean(fold_scores),
                "std_accuracy": np.std(fold_scores),
            }
        )

    param_check = pd.DataFrame(rows).sort_values("mean_accuracy", ascending=False).reset_index(drop=True)
    display(param_check)
else:
    print("Skipped. Set RUN_EXTRA_PARAM_CHECK = True to run the quick comparison.")


## 7. Optional Optuna Mini Search

The exact demo does not require Optuna because the winning parameters are fixed above. If `optuna` is installed, set `RUN_OPTUNA = True` to run a small classroom-friendly search.


In [ ]:
RUN_OPTUNA = True
N_TRIALS = 8

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        **XGB_BEST_PARAMS,
        "n_estimators": trial.suggest_int("n_estimators", 400, 900, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.08, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.85, 1.0, step=0.05),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.80, 1.0, step=0.05),
        "reg_alpha": trial.suggest_float("reg_alpha", 2.0, 7.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 5.0),
    }
    mean_acc, _ = cv_accuracy(params, X_model, y, n_splits=3)
    return mean_acc


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS)
print(f"Best mini-search CV ({N_TRIALS} trials): {study.best_value:.5f}")
print("Best mini-search params:")
print(study.best_params)


## 8. Final Training and Submission

KMeansSMOTE balances the compact pruned training matrix, then a single XGBoost model is fitted on all resampled rows.


In [ ]:
smote = KMeansSMOTE(sampling_strategy=1, n_jobs=-1, random_state=RANDOM_STATE)
X_train_final, y_train_final = smote.fit_resample(X_model, y)

print(f"After KMeansSMOTE: X={X_train_final.shape}")
print(y_train_final.value_counts().rename("class_count"))

final_model = XGBClassifier(**XGB_BEST_PARAMS)
final_model.fit(X_train_final, y_train_final)

test_pred = final_model.predict(test_model)
submission = sample_submission.copy()
submission["Transported"] = test_pred.astype(bool)
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH.relative_to(ROOT)}")
print(f"Predicted True count: {int(submission['Transported'].sum())} / {len(submission)}")
print(f"Predicted True rate: {submission['Transported'].mean():.4f}")
display(submission.head())


## 9. Demo Talking Points

- This is a leaderboard-oriented branch: local CV is modest, but the public score was strong for this competition.
- The most important reproducibility controls are fixed random seeds, explicit preprocessing, fixed XGB parameters, and a saved submission path.
- The optional tuning cell is intentionally small. For a formal report, use larger repeated CV or a held-out audit protocol; for a classroom demo, this version is fast and understandable.
